# PySpark DataFrame Data Engineering Project — Aviation Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `flights.csv`, `bookings.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("flights.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("bookings.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select flight_id and airline from the first DataFrame.

In [ ]:
df1.select('flight_id','airline').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where fare is greater than its average.

In [ ]:
avg_value = df2.select(avg('fare').alias('avg')).first()['avg']
df2.filter(col('fare') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on fare using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('fare') >= avg('fare').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in airline and travel_class with 'Unknown'.

In [ ]:
df1.fillna({'airline': 'Unknown'}).show(5)
df2.fillna({'travel_class': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using flight_id and flight_id.

In [ ]:
joined = df1.join(df2, df1['flight_id'] == df2['flight_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['flight_id'] == df2['flight_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average fare by travel_class.

In [ ]:
df2.groupBy('travel_class').agg(count('*').alias('records'), avg('fare').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total fare by travel_class.

In [ ]:
df2.groupBy('travel_class').agg(sum('fare').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum fare in each travel_class.

In [ ]:
df2.groupBy('travel_class').agg(max('fare').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum fare in each travel_class.

In [ ]:
df2.groupBy('travel_class').agg(min('fare').alias('min_value')).show()

## Q14. Having Condition

Show travel_class groups whose total fare is greater than the overall average fare.

In [ ]:
overall = df2.select(avg('fare').alias('v')).first()['v']
df2.groupBy('travel_class').agg(sum('fare').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by fare.

In [ ]:
df2.orderBy(desc('fare')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by fare.

In [ ]:
df2.orderBy(asc('fare')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by fare within each travel_class.

In [ ]:
w = Window.partitionBy('travel_class').orderBy(desc('fare'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of fare within each travel_class.

In [ ]:
w = Window.partitionBy('travel_class').orderBy('booking_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('fare').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's fare with the previous record in its travel_class.

In [ ]:
w = Window.partitionBy('travel_class').orderBy('booking_date')
df2.withColumn('previous_value', lag('fare').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each travel_class's contribution to total fare.

In [ ]:
tot = df2.select(sum('fare').alias('t')).first()['t']
df2.groupBy('travel_class').agg(sum('fare').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from booking_date.

In [ ]:
df2.withColumn('year', year('booking_date')).withColumn('month', month('booking_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average fare by month.

In [ ]:
df2.withColumn('month', date_format('booking_date','yyyy-MM')).groupBy('month').agg(sum('fare').alias('total'), avg('fare').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with travel_class as rows and year of booking_date as columns.

In [ ]:
df2.withColumn('year', year('booking_date')).groupBy('travel_class').pivot('year').agg(sum('fare')).show()

## Q24. Conditional KPI

Count records where fare is above the category average.

In [ ]:
w=Window.partitionBy('travel_class')
df2.withColumn('cat_avg',avg('fare').over(w)).filter(col('fare')>col('cat_avg')).groupBy('travel_class').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total fare by airline.

In [ ]:
df1.join(df2, df1['flight_id']==df2['flight_id']).groupBy(df1['airline']).agg(sum(df2['fare']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank airline by total fare.

In [ ]:
r=df1.join(df2,df1['flight_id']==df2['flight_id']).groupBy(df1['airline']).agg(sum(df2['fare']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for fare.

In [ ]:
df2.withColumn('segment',when(col('fare')<1000,'Low').when(col('fare')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative fare values.

In [ ]:
df2.filter(col('fare')<0).count()

## Q29. Final Business Report

Create a final report by travel_class with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('travel_class').agg(count('*').alias('records'),sum('fare').alias('total'),avg('fare').alias('average'),min('fare').alias('minimum'),max('fare').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project